# Activity: Software Development Kit for the National Weather Service API
In this activity, students will build a Software Development Kit (SDK) for the [National Weather Service API](https://www.weather.gov/documentation/services-web-api). We'll retrieve weather data for Ithaca, NY, using the `(latitude, longitude)` coordinates `42.443961` and `-76.501881`. We'll use [the `points` endpoint](https://www.weather.gov/documentation/services-web-api#/default/point) to get the grid data for this location and use this data for additional calls. 

> __Learning Objectives:__
> 
> By the end of this activity, you will be able to:
> * **Call RESTful APIs**: Construct and execute API calls to retrieve data from web services, understanding how URLs encode endpoints and parameters.
> * **Process structured API responses**: Navigate and extract data from nested JSON objects returned by APIs, understanding the tree structure of typical RESTful responses.
> * **Apply handler patterns for data transformation**: Use handler functions to process raw API responses into more convenient formats before presenting them to users.

In addition to demonstrating how to call an Application Programming Interface (API), the code that backs this example has some interesting design choices. Let's check them out!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [1]:
include("Include.jl");

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

### Constants
We'll define constants for the latitude and longitude of our target location. See the comments for additional details.

In [2]:
latitude = 42.443961; # latitude value for Ithaca, NY (change this for your target location)
longitude = -76.501881; # longitude value for Ithaca, NY (change this for your target location)

___

## Task 1: Build a GridPoint Endpoint Model and Call the Weather API
In this task, we'll create a model for the endpoint call and then retrieve raw weather data for Ithaca, NY (or another specified target location).

First, let's build [a `MyWeatherGridPointEndpointModel` instance](src/Types.jl) by passing in the `(latitude, longitude)` data as keyword arguments to the constructor method. [The `points` endpoint](https://www.weather.gov/documentation/services-web-api#/default/point) returns metadata about a given latitude/longitude point.

In [3]:
model = MyWeatherGridPointEndpointModel(latitude = latitude, longitude = longitude)

MyWeatherGridPointEndpointModel(42.443961, -76.501881)

Next, let's construct [the Uniform Resource Locator (URL) string](https://en.wikipedia.org/wiki/Query_string) using [a custom `build(...)` method](src/Factory.jl). This `URL` string will encode the resource we want (endpoint) and how to access it (protocol). We save the weather service `URL` in the `points_url_string::String` variable:

In [4]:
points_url_string = build("https://api.weather.gov", model)

"https://api.weather.gov/points/42.443961,-76.501881"

Now, let's make our first call! Notice the `syntax sugar` we used to make it look like our endpoint model is a function. 

> __What is happening behind the scenes__? The code creates [an anonymous function](https://docs.julialang.org/en/v1/manual/functions/#man-anonymous-functions) that takes the `model::MyWeatherGridPointEndpointModel` instance and `points_url_string::String` variable, makes the network call, and returns the data.

We store the data returned from the API call in the `result_points::Dict{String, Any}` variable:

In [5]:
result_points = MyWeatherGridPointEndpointModel(points_url_string) # Hmmm. That's cool - a type as a function.

JSON.Object{String, Any} with 5 entries:
  "@context"   => Any["https://geojson.org/geojson-ld/geojson-context.jsonld", …
  "id"         => "https://api.weather.gov/points/42.444,-76.5019"
  "type"       => "Feature"
  "geometry"   => Object{String, Any}("type"=>"Point", "coordinates"=>Any[-76.5…
  "properties" => Object{String, Any}("@id"=>"https://api.weather.gov/points/42…

We can access the data in the dictionary using standard Julia dictionary syntax. For example, we can get the `properties` field from the returned data like this:

In [6]:
result_points["properties"] # Access the 'properties' field from the returned data

JSON.Object{String, Any} with 17 entries:
  "@id"                 => "https://api.weather.gov/points/42.444,-76.5019"
  "@type"               => "wx:Point"
  "cwa"                 => "BGM"
  "forecastOffice"      => "https://api.weather.gov/offices/BGM"
  "gridId"              => "BGM"
  "gridX"               => 44
  "gridY"               => 70
  "forecast"            => "https://api.weather.gov/gridpoints/BGM/44,70/foreca…
  "forecastHourly"      => "https://api.weather.gov/gridpoints/BGM/44,70/foreca…
  "forecastGridData"    => "https://api.weather.gov/gridpoints/BGM/44,70"
  "observationStations" => "https://api.weather.gov/gridpoints/BGM/44,70/statio…
  "relativeLocation"    => Object{String, Any}("type"=>"Feature", "geometry"=>O…
  "forecastZone"        => "https://api.weather.gov/zones/forecast/NYZ025"
  "county"              => "https://api.weather.gov/zones/county/NYC109"
  "fireWeatherZone"     => "https://api.weather.gov/zones/fire/NYZ210"
  "timeZone"            => "America/

The [National Weather Service API](https://www.weather.gov/documentation/services-web-api) returns the forecast URL directly from the endpoint call, which we can extract and store in the `forecast_url::String` variable.
> __Is this normal__? No, this is strange because the fully formed forecast `URL` is returned directly. Usually, we'd need to get some intermediate data from the `endpoint` call and then package it in the `URL` string ourselves. However, each API is unique in design.

Let's grab [the URL string](https://en.wikipedia.org/wiki/Query_string) for hourly forecast data from the `result_points` dictionary like this:

In [7]:
forecast_url = result_points["properties"]["forecastHourly"]

"https://api.weather.gov/gridpoints/BGM/44,70/forecast/hourly"

Finally, we have [the URL string](https://en.wikipedia.org/wiki/Query_string), allowing us to get the weather forecast for Ithaca, NY. Let's call the [Weather Service API](https://www.weather.gov/documentation/services-web-api) again with the `forecast_url::String` and look at what comes back from the API:

In [8]:
result_forecast_raw = MyWeatherForecastEndpointModel(forecast_url) # notice the pattern - check out the code to see what is going on

JSON.Object{String, Any} with 4 entries:
  "@context"   => Any["https://geojson.org/geojson-ld/geojson-context.jsonld", …
  "type"       => "Feature"
  "geometry"   => Object{String, Any}("type"=>"Polygon", "coordinates"=>Any[Any…
  "properties" => Object{String, Any}("units"=>"us", "forecastGenerator"=>"Hour…

The forecast data comes back as a vector of JSON objects. The forecast data is stored in the `properties` field, in a sub-field called `periods`. 

> __What do we see__? The forecast data is a list of JSON objects, each containing forecast data for a specific time period. Each JSON object contains fields like, `startTime`, `isDaytime`, `temperature`, `windSpeed`, etc. There are also some nested objects, for example, the `probabilityOfPrecipitation` field. Thus, it is a tree of data, which is common for RESTful APIs. However, this raw data is not very convenient to work with.

Let's examine this data:

In [9]:
result_forecast_raw["properties"]["periods"] # Access the 'periods' field from the returned forecast data

156-element Vector{Any}:
 JSON.Object{String, Any}("number" => 1, "name" => "", "startTime" => "2025-11-21T05:00:00-05:00", "endTime" => "2025-11-21T06:00:00-05:00", "isDaytime" => false, "temperature" => 36, "temperatureUnit" => "F", "temperatureTrend" => nothing, "probabilityOfPrecipitation" => JSON.Object{String, Any}("unitCode" => "wmoUnit:percent", "value" => 1), "dewpoint" => JSON.Object{String, Any}("unitCode" => "wmoUnit:degC", "value" => 1.6666666666666667)…)
 JSON.Object{String, Any}("number" => 2, "name" => "", "startTime" => "2025-11-21T06:00:00-05:00", "endTime" => "2025-11-21T07:00:00-05:00", "isDaytime" => true, "temperature" => 36, "temperatureUnit" => "F", "temperatureTrend" => nothing, "probabilityOfPrecipitation" => JSON.Object{String, Any}("unitCode" => "wmoUnit:percent", "value" => 4), "dewpoint" => JSON.Object{String, Any}("unitCode" => "wmoUnit:degC", "value" => 1.6666666666666667)…)
 JSON.Object{String, Any}("number" => 3, "name" => "", "startTime" => "2025-11-2

While this raw data is interesting, it is not convenient to work with. We can improve this by building a handler to process the data before returning it.

___

## Task 2: Let's do a better job at handling the forecast data coming back
In this task, we'll demonstrate a handler pattern in which the raw data from the API is processed __before__ being presented to the caller. 

> __Why would we do this?__ We use a handler pattern if we want the data from the API to be in a more convenient form, for example [as a DataFrame instance](https://dataframes.juliadata.org/stable/), or we want to manipulate or process the data in some way. Ultimately, using this pattern makes our lives easier when using the API data.

Notice the __syntactic sugar__ pattern again. We call the `MyWeatherForecastEndpointModel` instance as a function, passing the `forecast_url::String` variable and a handler function `process_forecast_response_dataframe(...)`.

In [10]:
result_forecast = MyWeatherForecastEndpointModel(forecast_url, handler = process_forecast_response_dataframe) # Notice the handler pattern

Row,startTime,endTime,isDayTime,temperature,temperatureUnit,windSpeed,windDirection,shortForecast
,String,String,Bool,Int64,String,String,String,String
1,2025-11-21T05:00:00-05:00,2025-11-21T06:00:00-05:00,false,36,F,3 mph,SE,Mostly Cloudy
2,2025-11-21T06:00:00-05:00,2025-11-21T07:00:00-05:00,true,36,F,3 mph,SE,Mostly Cloudy
3,2025-11-21T07:00:00-05:00,2025-11-21T08:00:00-05:00,true,37,F,5 mph,SE,Mostly Cloudy
4,2025-11-21T08:00:00-05:00,2025-11-21T09:00:00-05:00,true,39,F,6 mph,S,Slight Chance Rain Showers
5,2025-11-21T09:00:00-05:00,2025-11-21T10:00:00-05:00,true,43,F,6 mph,S,Partly Sunny
6,2025-11-21T10:00:00-05:00,2025-11-21T11:00:00-05:00,true,46,F,6 mph,S,Partly Sunny
7,2025-11-21T11:00:00-05:00,2025-11-21T12:00:00-05:00,true,50,F,8 mph,SW,Mostly Cloudy
8,2025-11-21T12:00:00-05:00,2025-11-21T13:00:00-05:00,true,51,F,8 mph,SW,Mostly Cloudy
9,2025-11-21T13:00:00-05:00,2025-11-21T14:00:00-05:00,true,52,F,8 mph,SW,Mostly Cloudy


Now the data comes back as a [DataFrame instance](https://dataframes.juliadata.org/stable/), which is much easier to work with! However, we haven't returned all the data, only the fields we care about.

> __How does this work__? Behind the scenes, we create an anonymous function that maps the `model::MyWeatherForecastEndpointModel` instance, the `forecast_url::String` variable, and the `process_forecast_response_dataframe(...)` function to a private function, which makes the network call, processes the data using the handler function, and returns the processed data.

This approach is very flexible. We can build different handler functions to process the data in different ways, depending on our needs. However, if we are using code someone else wrote, we need to understand how the handler pattern works, and what handler functions are available to us.

___

## Summary
In this activity, we built an SDK to interact with the National Weather Service API, demonstrating core patterns for working with RESTful web services.

> __Key Takeaways:__
>
> 1. **Endpoint chaining in APIs**: Many APIs require multiple calls where data from one response feeds into subsequent requests. The National Weather Service returns a grid point endpoint URL that we use for forecasts, showing how APIs are often structured as chains of dependent calls.
> 2. **Raw responses are not always convenient**: JSON responses from APIs often contain more data than we need and in nested structures that are difficult to work with. This motivates using handler functions to transform raw API responses into more usable formats like DataFrames.
> 3. **Handler functions separate concerns**: By applying handlers at the API call level, we separate data retrieval from data processing, making the SDK more flexible. Different users can apply different handlers depending on their needs without modifying the core API calling code.

___